In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

## Install libraries

```bash
conda create -n edu4 python=3.11 jupyter matplotlib
```

```bash 
! pip install -U -r requirements.txt
```

```bash
! pip install -U numpy
! pip install -U scikit-learn
```

## Update repository

In [ ]:
# ! git pull

## Add import path

In [ ]:
import gc
import os
import sys
from pathlib import Path

In [ ]:
def add_library_level(level=4):
    suf_path = ['..']
    path = '..'
    for i in range(0, level):
        join_path = suf_path * i
        path = '/'.join(join_path)
        module_path = os.path.abspath(os.path.join(path))
        if module_path not in sys.path:
            sys.path.append(module_path)
            print(f'Appendeding {path}')

In [ ]:
add_library_level(level=5)

## Organize imports

In [ ]:
import multiprocessing

In [ ]:
from src.lattmc.utils import init_device, empty_cache

In [ ]:
from src.lattmc.fca.utils import *
from src.lattmc.fca.lattice_utils import diff
from src.lattmc.fca.fca_utils import *

In [ ]:
from src.lattmc.tc.transcoder_analyzers import ConceptAnalysis, init_analyzer

In [ ]:
from src.lattmc.sae.nlp_sae_utils import gen_concept

In [ ]:
import logging

#### Number of CPU cores

In [ ]:
workers = multiprocessing.cpu_count()
workers

In [ ]:
SEED = 2026

In [ ]:
logging.basicConfig(level=logging.INFO)

In [ ]:
device = init_device()
device

In [ ]:
torch.__version__

In [ ]:
np.__version__

In [ ]:
# np.set_printoptions(precision=4, suppress=True)

## Initialize Paths

In [ ]:
OWT_PATH = Path().resolve().parents[1]
PATH = OWT_PATH / 'notebooks' / 'transcoders' / 'data'
GPT2 = PATH / 'transcoders' / 'gpt2'
OWT_TOKENS_DIR = GPT2 / 'owt_tokens'
TOKENS_PATH = OWT_TOKENS_DIR / 'owt_tokens_torch.pt'
OWT_TOKENS_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
! ls {GPT2}

In [ ]:
GPT2.absolute()

In [ ]:
! ls {TOKENS_PATH}

In [ ]:
! ls {str(TOKENS_PATH.absolute())}

## Load trancoders

In [ ]:
layers = list(range(12))
layers = [0, 4, 6, 8, 10, 11]

In [ ]:
tr_analyzer = init_analyzer(
    layers,
    TOKENS_PATH,
    GPT2,
    model_name='gpt2-small',
    device=device,
    tr_or_sae=True,
    vector_dir=GPT2,
)

In [ ]:
empty_cache()

In [ ]:
tr_analyzer.transcoder.model.cfg.device

In [ ]:
empty_cache()

## Features Dimension

In [ ]:
tr_analyzer.fcas[0].V.shape, tr_analyzer.fcas[8].V.shape, tr_analyzer.fcas[11].V.shape

## Analyze Tokens

In [ ]:
tr_analyzer.tokens.shape

In [ ]:
tr_analyzer.tokens[0][0]

In [ ]:
tn = to_numpy(tr_analyzer.tokens[0][0])

In [ ]:
tn.item()

## New York City

In [ ]:
text = ' New York City'
text

In [ ]:
text_id = 3457
text = tr_analyzer.texts[text_id]
prompt = tr_analyzer.corpus[text_id]
text

In [ ]:
concept_an = ConceptAnalysis(prompt, tr_analyzer)

In [ ]:
concept_an.corpus[text_id]

In [ ]:
concept_an.analyze_concepts()

In [ ]:
t_idcs = [2, 3]
t_idcs = [1]
# t_idcs = [2]
t_idcs = [1, 2, 3]

### New York City layer 0 

In [ ]:
layer = 0

In [ ]:
A_ny = dict()

In [ ]:
tr_analyzer.to_string(prompt[3])

In [ ]:
t_new = prompt[1]
t_york = prompt[2]
t_city = prompt[3]
tr_analyzer.transcoder.tokenizer.decode(t_new), tr_analyzer.transcoder.tokenizer.decode(t_york), \
tr_analyzer.transcoder.tokenizer.decode(t_city)

In [ ]:
t_york

In [ ]:
tr_analyzer.transcoder.tokenizer.decode(prompt[t_idcs])

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, red_val=[10.8, 26.8, 18.8], limit=100, min_vals=True, full_tokens=True,)

#### Calculate $G_{I}F_{I}$

In [ ]:
layer = 0

In [ ]:
tr_analyzer.corpus.shape[0]

In [ ]:
A_nyc = np.random.randint(0, high=tr_analyzer.corpus.shape[0], size=11)
A_nyc

In [ ]:
A_nyc = np.array(
    [25576, 
     19744,  
     4488,  
     8217, 
     21048, 
     11717, 
     15774,  
     1431, 
     19151,
     15355, 
     24614
])

In [ ]:
A_nyc = np.array([17525, 11882,  1305,  2024, 19824,  7857,  5753, 20095, 22451,
         841, 10946])

In [ ]:
A_nyc = np.array([11882, 1305, 7857, 20095, 22451, 841, 10946])

In [ ]:
A_nyc = np.array([1305, 7857, 22451, 841, 10946])

In [ ]:
A_nyc = np.array([25314,  7705, 25413,  1757,  9276, 19655, 23050, 18379, 19094,
        8099, 17518, 13766, 20164,  2949, 15098, 11772, 23708])

In [ ]:
# A_nyc = np.array([
#     2044, 
#     3574,
#     3215,
#     4993,
#     2835,
#     117,
#     25520,
#     1,
#     2,
#     3,
#     4,
#     5,
# ])
conA = tr_analyzer.fcas[layer].GF_F(A_nyc)

In [ ]:
val, idcs = topNonZeros(conA.v)
val, idcs

In [ ]:
val.shape

In [ ]:
A_nyc.shape, conA.A.shape

In [ ]:
A_nyc

In [ ]:
p_k = topRangeProject(conA.v, from_idx=18, to_idx=20)
topK(p_k, k=5)

In [ ]:
tr_analyzer.print_all_from_objects(
    layer,
    A_nyc,
    p_k,
    with_text=True,
    limit=185,
)

In [ ]:
tr_analyzer.print_all_from_objects(
    layer,
    conA.A,
    p_k,
    with_text=True,
    limit=185,
)

#### Initialize Minumal FCA on Layer 0

In [ ]:
A_ny = dict()
A_ny[layer] = concept_an.c_is[layer].A

In [ ]:
V_nonz = np.minimum(concept_an.fcas[layer].V, concept_an.fcas[layer].v_min_nonzeros)
concept_an.fcas[layer].V.shape, V_nonz.shape

In [ ]:
nonzFCA = NonzeroFCA.fromFCA(concept_an.fcas[layer])

In [ ]:
nz_analyzer = tr_analyzer.fromAnalyzer(
    tr_analyzer,
    fcas = {
        layer: nonzFCA,
    }
    fca_vecs
)

In [ ]:
nz_analyzer.fcas

In [ ]:
concept_nz = ConceptAnalysis(prompt, nz_analyzer)

In [ ]:
concept_nz.analyze_concepts()

In [ ]:
concept_nz.gen_text(t_idcs, layer, red_val=[4, 4, 4], limit=100, min_vals=True, full_tokens=True,)

In [ ]:
np.all(nonzFCA.V == V_nonz)

In [ ]:
A_nonz = nonzFCA.GF(concept_an.c_is[layer].A)

In [ ]:
np.any(intersect(A_nonz, concept_an.c_is[layer].A) == concept_an.c_is[layer].A)

In [ ]:
A_ny[layer].shape

In [ ]:
# intersect(A_ny[0], A_ny[8]).shape

In [ ]:
vs = concept_an.transcoder('New York City', layer)

In [ ]:
vs.shape

In [ ]:
concept_an.c_is

In [ ]:
A_ny[layer] = concept_an.c_is[layer].A

In [ ]:
concept_an.c_is[layer].A.shape

In [ ]:
t_idcs

In [ ]:
u = concept_an.vs[layer]
u_12 = meet(u[1], u[2])
u_23 = meet(u[2], u[3])
u_13 = meet(u[1], u[3])
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
u_23_mins = tr_analyzer.min_nonzero_vals(layer, u_23)
u_13_mins = tr_analyzer.min_nonzero_vals(layer, u_13)

In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer, with_text=True, limit=100)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
F_A_new = c.c.v

In [ ]:
topNonZeros(c.v), topNonZeros(c.c.v)

In [ ]:
c.c.A.shape

In [ ]:
tr_analyzer.texts[485]

In [ ]:
vs = tr_analyzer.transcoder(tr_analyzer.corpus[485], layer)[0]
u_new1 = vs[125]
u_new1.shape, tr_analyzer.transcoder.decode(tr_analyzer.corpus[485][125])

In [ ]:
vs = tr_analyzer.transcoder(tr_analyzer.corpus[353], layer)[0]
u_new2 = vs[44]
u_new2.shape, tr_analyzer.transcoder.decode(tr_analyzer.corpus[353][44])

In [ ]:
vs = tr_analyzer.transcoder(tr_analyzer.corpus[549], layer)[0]
u_new3 = vs[54]
u_new3.shape, tr_analyzer.transcoder.decode(tr_analyzer.corpus[549][54])

In [ ]:
vs = tr_analyzer.transcoder(tr_analyzer.corpus[900], layer)[0]
u_new4 = vs[85]
tk = tr_analyzer.corpus[900][85],
u_new4.shape, tk, tr_analyzer.transcoder.decode(tr_analyzer.corpus[900][85])

In [ ]:
vs = tr_analyzer.transcoder(tr_analyzer.corpus[1656], layer)[0]
u_new5 = vs[16]
tk = tr_analyzer.corpus[1656][16]
u_new5.shape, tk, tr_analyzer.transcoder.decode(tr_analyzer.corpus[1656][16])

In [ ]:
new_list = [u[1], u_new1, u_new2, u_new3, u_new4, u_new5]

In [ ]:
tau_new = meet_all(new_list)
topNonZeros(tau_new)

In [ ]:
tau_new_fc = meet(tau_new, F_A_new)
topNonZeros(tau_new_fc)

In [ ]:
tau_new[22743]

In [ ]:
tau_zer = np.zeros_like(tau_new)
tau_zer[22743] = tau_new[22743]

In [ ]:
w_new = meet(tau_zer, tr_analyzer.fcas[layer].v_min_nonzeros)
vals, idcs = topNonZeros(w_new)
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
c.c.A.shape

In [ ]:
with tqdm(tr_analyzer.corpus) as pcorpus:
    G_T_new_list = [idx for idx, pr in enumerate(pcorpus) if t_new in pr]
G_T_new = np.array(G_T_new_list)

In [ ]:
G_T_new.shape

In [ ]:
c_new = tr_analyzer.fcas[layer].GF_F(G_T_new)

In [ ]:
c_new

In [ ]:
diff_FC = diff(c_new.A, G_T_new)

In [ ]:
print(tr_analyzer.texts[diff_FC])

In [ ]:
for idx in diff_FC:
    pr = tr_analyzer.corpus[idx]
    for t_p in pr:
        if t_new == t_p:
            print(tr_analyzer.transcoder.decode(t_p))
        if 'new' in tr_analyzer.transcoder.decode(t_p) or 'New' in tr_analyzer.transcoder.decode(t_p):
            print(tr_analyzer.transcoder.decode(t_p), idx)

In [ ]:
topNonZeros(c_new.v)

In [ ]:
df_new = diff(G_T_new, c.c.A)
df_new.shape, G_T_new.shape[0] - c.c.A.shape[0]

In [ ]:
np.intersect1d(G_T_new, c.c.A).shape[0] / np.union1d(G_T_new, c.c.A).shape[0]

In [ ]:
mt_new = np.intersect1d(G_T_new, c.c.A)
mt_new.shape

In [ ]:
df_idcs = dict()
with tqdm(df_new) as pdfs:
    for k in pdfs:
        pr = tr_analyzer.corpus[k]
        df_idcs.setdefault(k, list())
        for idx, tk in enumerate(pr) :
            if tk == t_new:
                df_idcs[k].append(idx)

In [ ]:
df_idcs

In [ ]:
vd = tr_analyzer.transcoder(tr_analyzer.corpus[df_new], layer)

In [ ]:
vd.shape

In [ ]:
us_new = list()
with tqdm(df_idcs.items()) as pdf_items:
    for idx, (k, df_ids) in enumerate(pdf_items):
        vd_i = vd[idx]
        vd_new = vd_i[df_ids][0]
        tks = tr_analyzer.corpus[k][df_ids]
        print(tr_analyzer.transcoder.decode(tks))
        print(le(w_new, vd_new))
        print(k, vd_i.shape, vd_new.shape)
        us_new.append(vd_new)

In [ ]:
us_new

In [ ]:
us_new_meet = meet_all(us_new)
us_new_meet

In [ ]:
topNonZeros(us_new_meet)

In [ ]:
tau_new_x = meet(tau_new, us_new_meet)
topNonZeros(tau_new_x)

In [ ]:
print(tr_analyzer.texts[df_new])

In [ ]:
w_new_x = meet(tau_new_x, tr_analyzer.fcas[layer].v_min_nonzeros)
vals, idcs = topNonZeros(w_new_x)
vals, idcs

In [ ]:
ts_new = tr_analyzer.tokenize('new')
ts_new = ts_new[0][1:].unsqueeze(0)
tr_analyzer.to_string(ts_new)

In [ ]:
c_new = tr_analyzer.transcoder(ts_new, layer)[0]

In [ ]:
ts_new = tr_analyzer.tokenize('New')
ts_new = ts_new[0][1:].unsqueeze(0)
tr_analyzer.to_string(ts_new)

In [ ]:
s_new = tr_analyzer.transcoder(ts_new, layer)[0]

In [ ]:
tr_analyzer.transcoder.decode(ts_new[0][1]), ts_new[0][1]

In [ ]:
ts_new = tr_analyzer.tokenize(' New')
ts_new = ts_new[0][1:].unsqueeze(0)
tr_analyzer.to_string(ts_new)

In [ ]:
r_new = tr_analyzer.transcoder(ts_new, layer)[0]

In [ ]:
tr_analyzer.transcoder.decode(ts_new[0][1]), ts_new[0][1]

In [ ]:
topNonZeros(c_new[1]), topNonZeros(s_new[1]), topNonZeros(r_new[1])

In [ ]:
tau_new_s = meet_all([tau_new_x, s_new[1], c_new[1], r_new[1]])

In [ ]:
topNonZeros(tau_new_s)

In [ ]:
s_new.shape

In [ ]:
w_new_s = meet(tau_new_s, tr_analyzer.fcas[layer].v_min_nonzeros)
vals, idcs = topNonZeros(w_new_s)
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=150)

In [ ]:
c.c.A.shape

In [ ]:
df_new = diff(G_T_new, c.c.A)
df_new.shape, G_T_new.shape[0] - c.c.A.shape[0]

In [ ]:
tr_analyzer.texts[df_new]

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
concept_an.G_FG

### New York City layer 8

In [ ]:
layer = 8

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, red_val=[4, 4, 4], limit=100, min_vals=True, full_tokens=True,)

#### Calculate $G_{I}F_{I}$

In [ ]:
layer = 8

In [ ]:
A_nyc = np.array([17525, 11882,  1305,  2024, 19824,  7857,  5753, 20095, 22451,
         841, 10946])

In [ ]:
A_nyc = np.array([25314,  7705, 25413,  1757,  9276, 19655, 23050, 18379, 19094,
        8099, 17518, 13766, 20164,  2949, 15098, 11772, 23708])

In [ ]:
A_nyc = np.array([11882, 1305, 7857, 20095, 22451, 841, 10946])

In [ ]:
# A_nyc = np.array([
#     2044, 
#     3574,
#     3215,
#     4993,
#     2835,
# ])
conA = tr_analyzer.fcas[layer].GF_F(A_nyc)

In [ ]:
val, idcs = topNonZeros(conA.v)
val, idcs

In [ ]:
val.shape

In [ ]:
A_nyc.shape, conA.A.shape

In [ ]:
A_nyc

In [ ]:
p_k = topRangeProject(conA.v, from_idx=5, to_idx=6)
topK(p_k, k=5)

In [ ]:
tr_analyzer.print_all_from_objects(
    layer,
    A_nyc,
    p_k,
    with_text=True,
    limit=185,
)

In [ ]:
tr_analyzer.print_all_from_objects(
    layer,
    conA.A,
    p_k,
    with_text=True,
    limit=185,
)

#### Initialize Minumal FCA on Layer 8 

In [ ]:
A_ny = dict()
A_ny[layer] = concept_an.c_is[layer].A

In [ ]:
V_nonz = np.minimum(concept_an.fcas[layer].V, concept_an.fcas[layer].v_min_nonzeros)
concept_an.fcas[layer].V.shape, V_nonz.shape

In [ ]:
nonzFCA = NonzeroFCA.fromFCA(concept_an.fcas[layer])

In [ ]:
nz_analyzer = tr_analyzer.fromAnalyzer(
    tr_analyzer,
    fcas = {
        layer: nonzFCA,
    }
)

In [ ]:
nz_analyzer.fcas

In [ ]:
concept_nz = ConceptAnalysis(prompt, nz_analyzer)

In [ ]:
concept_nz.analyze_concepts()

In [ ]:
concept_nz.gen_text(t_idcs, layer, red_val=[4, 4, 4], limit=100, min_vals=True, full_tokens=True,)

In [ ]:
np.all(nonzFCA.V == V_nonz)

In [ ]:
A_nonz = nonzFCA.GF(concept_an.c_is[layer].A)

In [ ]:
np.any(intersect(A_nonz, concept_an.c_is[layer].A) == concept_an.c_is[layer].A)

In [ ]:
A_ny[layer].shape

In [ ]:
# intersect(A_ny[0], A_ny[8]).shape

In [ ]:
u = concept_an.vs[layer]
u_12 = meet(u[1], u[2])
u_23 = meet(u[2], u[3])
u_13 = meet(u[1], u[3])
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
u_23_mins = tr_analyzer.min_nonzero_vals(layer, u_23)
u_13_mins = tr_analyzer.min_nonzero_vals(layer, u_13)

In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs[2:3], vals[2:3], layer, with_text=True, limit=100)

In [ ]:
u = concept_an.vs[layer]
u12 = meet(u[t_idcs[0]], u[t_idcs[1]])
u23 = meet(u[t_idcs[1]], u[t_idcs[2]])
u13 = meet(u[t_idcs[0]], u[t_idcs[2]])
min_12 = tr_analyzer.min_nonzero_vals(layer, u12)
min_23 = tr_analyzer.min_nonzero_vals(layer, u23)
min_13 = tr_analyzer.min_nonzero_vals(layer, u13)

In [ ]:
vals, idcs = topK(u_12, k=u_12.shape[0])

In [ ]:
vals, idcs = topK(min_12, k=min_12.shape[0])
vals, idcs

In [ ]:
if np.any(u_12 > 0):
   _, c = concept_an.gen_and_print(idcs[2:3], vals[2:3], layer, with_text=True, limit=100)

In [ ]:
np.any(u23 > 0)

In [ ]:
vals, idcs = topNonZeros(min_23)
vals, idcs

In [ ]:
if np.any(u23 > 0):
    _, c = concept_an.gen_and_print(idcs[2:3], vals[2:3], layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_13_mins)
vals, idcs

In [ ]:
empty_cache()

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals[:10], idcs[:10]

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
if np.any(vals[1:] > 0):
    _, c = concept_an.gen_and_print(idcs[1:], vals[1:], layer, with_text=True, limit=100)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print([16679, 10130, 20994], [12, 9, 9], layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

### New York City layer 11

In [ ]:
layer = 11

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, red_val=[4, 4, 4], limit=100, set_val=0.2, min_vals=True, full_tokens=True,)

#### Calculate $G_{I}F_{I}$

In [ ]:
layer = 11

In [ ]:
A_nyc = np.array([17525, 11882,  1305,  2024, 19824,  7857,  5753, 20095, 22451,
         841, 10946])

In [ ]:
A_nyc_2 = np.random.randint(0, high=tr_analyzer.corpus.shape[0], size=17)
A_nyc = A_nyc_2
A_nyc

In [ ]:
A_nyc = np.array([25314,  7705, 25413,  1757,  9276, 19655, 23050, 18379, 19094,
        8099, 17518, 13766, 20164,  2949, 15098, 11772, 23708])

In [ ]:
A_nyc = np.array([11882, 1305, 7857, 20095, 22451, 841, 10946])

In [ ]:
# A_nyc = np.array([
#     2044, 
#     3574,
#     3215,
#     4993,
#     2835,
# ])
conA = tr_analyzer.fcas[layer].GF_F(A_nyc)

In [ ]:
val, idcs = topNonZeros(conA.v)
val, idcs

In [ ]:
val.shape

In [ ]:
A_nyc.shape, conA.A.shape

In [ ]:
A_nyc

In [ ]:
p_k = topRangeProject(conA.v, from_idx=12, to_idx=13)
topK(p_k, k=5)

In [ ]:
tr_analyzer.print_all_from_objects(
    layer,
    A_nyc,
    p_k,
    with_text=True,
    limit=185,
)

In [ ]:
tr_analyzer.print_all_from_objects(
    layer,
    conA.A,
    p_k,
    with_text=True,
    limit=185,
)

#### Initialize Minumal FCA on Layer 11

In [ ]:
A_ny = dict()
A_ny[layer] = concept_an.c_is[layer].A

In [ ]:
V_nonz = np.minimum(concept_an.fcas[layer].V, concept_an.fcas[layer].v_min_nonzeros)
concept_an.fcas[layer].V.shape, V_nonz.shape

In [ ]:
nonzFCA = NonzeroFCA.fromFCA(concept_an.fcas[layer])

In [ ]:
nz_analyzer = tr_analyzer.fromAnalyzer(
    tr_analyzer,
    fcas = {
        layer: nonzFCA,
    }
)

In [ ]:
nz_analyzer.fcas

In [ ]:
concept_nz = ConceptAnalysis(prompt, nz_analyzer)

In [ ]:
concept_nz.analyze_concepts()

In [ ]:
concept_nz.gen_text(t_idcs, layer, red_val=[4, 4, 4], limit=100, min_vals=True, full_tokens=True,)

In [ ]:
np.all(nonzFCA.V == V_nonz)

In [ ]:
A_nonz = nonzFCA.GF(concept_an.c_is[layer].A)

In [ ]:
np.any(intersect(A_nonz, concept_an.c_is[layer].A) == concept_an.c_is[layer].A)

In [ ]:
A_ny[layer].shape

In [ ]:
# intersect(A_ny[0], A_ny[8]).shape

In [ ]:
A_ny[layer] = concept_an.c_is[layer].A

In [ ]:
np.setdiff1d(A_ny[0], intersect(A_ny[0], A_ny[layer])).shape

In [ ]:
np.setdiff1d(A_ny[layer], intersect(A_ny[0], A_ny[layer])).shape

In [ ]:
concept_an.corpus[concept_an.c_is[layer].A].shape

In [ ]:
concept_an.det_tokens[layer].shape

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
# c = concept_an.gen_and_print([2094, 10988, 10533], [0.38, 0.30, 0.23], layer, with_text=True, limit=10)

In [ ]:
if np.any(vals[1:2] > 0):
    _, c = concept_an.gen_and_print(idcs[1:2], vals[1:2], layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=20)

In [ ]:
vs_0 = concept_an.vs[0][t_idcs]
vs_8 = concept_an.vs[8][t_idcs]
vs_11 = concept_an.vs[11][t_idcs]

In [ ]:
t_idcs

In [ ]:
i_idx = 2

In [ ]:
vs_0.shape

In [ ]:
topNonZeros(join(vs_0[i_idx], vs_8[i_idx]))

In [ ]:
topNonZeros(join(vs_0[i_idx], vs_11[i_idx]))

In [ ]:
topNonZeros(join(vs_8[i_idx], vs_11[i_idx]))

## Richmond Hill

In [ ]:
text_idx = 117
text = tr_analyzer.texts[text_idx]
prompt = tr_analyzer.corpus[text_idx]
text

In [ ]:
concept_an = ConceptAnalysis(prompt, tr_analyzer)

In [ ]:
concept_an.analyze_concepts()

In [ ]:
t_idcs = [2, 3]
t_idcs = [1]
# t_idcs = [2]
# t_idcs = [6, 13, 30, 31]
# t_idcs = [6, 30, 31]
t_idcs = [30, 31]

### Richmond Hill at Layer 0

In [ ]:
layer = 0

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=0.2, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
# v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

### Richmond Hill at Layer 8

In [ ]:
layer = 8

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=4, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
gc.collect()

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals[:10], idcs[:10]

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
if np.any(vals[1:] > 0):
    concept_an.gen_and_print(idcs[1:], vals[1:], layer, with_text=True, limit=10)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
# v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print([16679, 10130, 20994], [12, 9, 9], layer, with_text=True, limit=100)

### Richmond Hill at Layer 11

In [ ]:
layer = 11

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=4, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
# v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=12)

## National Park Service

In [ ]:
text_idx = 21481
text = tr_analyzer.texts[text_idx]
prompt = tr_analyzer.corpus[text_idx]
text

In [ ]:
concept_an = ConceptAnalysis(prompt, tr_analyzer)

In [ ]:
concept_an.analyze_concepts()

In [ ]:
t_idcs = [2, 3]
t_idcs = [1]
# t_idcs = [2]
t_idcs = [24, 25, 26]

### National Park Service at Layer 0

In [ ]:
layer = 0

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=0.2, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

### National Park Service at Layer 8

In [ ]:
layer = 8

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=0.2, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals[:10], idcs[:10]

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
if np.any(vals[1:] > 0):
    concept_an.gen_and_print(idcs[1:], vals[1:], layer, with_text=True, limit=100)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print([16679, 10130, 20994], [12, 9, 9], layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

### National Park Service at Layer 11

In [ ]:
layer = 11

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=0.2, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
# concept_an.gen_and_print([1944, 8298, 15520], [0.2, 0.2, 0.2], layer, with_text=True, limit=100)

In [ ]:
# if np.any(vals[1:2] > 0):
#     concept_an.gen_and_print(idcs[1:2], vals[1:2], layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

## Rio de Janeiro

In [ ]:
text_idx = 5411
text = tr_analyzer.texts[text_idx]
prompt = tr_analyzer.corpus[text_idx]
text

In [ ]:
concept_an = ConceptAnalysis(prompt, tr_analyzer)

In [ ]:
concept_an.analyze_concepts()

In [ ]:
t_idcs = [2, 3]
t_idcs = [1]
# t_idcs = [2]
t_idcs = [15, 26, 17]

### Rio de Janeiro on Layer 0

In [ ]:
layer = 0

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=None, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

### Rio de Janeiro on Layer 8

In [ ]:
layer = 8

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=None, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals[:10], idcs[:10]

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
if np.any(vals[1:] > 0):
    concept_an.gen_and_print(idcs[1:], vals[1:], layer, with_text=True, limit=10)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print([16679, 10130, 20994], [12, 9, 9], layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

### Rio de Janeiro at Layer 11

In [ ]:
layer = 11

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=4, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0): 
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
if np.any(vals[1:2] > 0):
    c = concept_an.gen_and_print(idcs[1:2], vals[1:2], layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=12)

## Sport Teams Cleveland, Clippers, Cavaliers

In [ ]:
text_idx = 1924
text = tr_analyzer.texts[text_idx]
prompt = tr_analyzer.corpus[text_idx]
text

In [ ]:
concept_an = ConceptAnalysis(prompt, tr_analyzer)

In [ ]:
concept_an.analyze_concepts()

In [ ]:
t_idcs = [2, 3]
t_idcs = [1]
# t_idcs = [2]
t_idcs = [15, 39, 94]

### Sports Teams Cleveland, Clippers, Cavaliers at Layer 0

In [ ]:
layer = 0

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=0.2, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

### Sports Teams Cleveland, Clippers, Cavaliers at Layer 8

In [ ]:
layer = 8

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=0.2, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals[:10], idcs[:10]

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
if np.any(vals[1:] > 0):
    c = concept_an.gen_and_print(idcs[1:], vals[1:], layer, with_text=True, limit=10)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print([16679, 10130, 20994], [12, 9, 9], layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

### Sports Teams Cleveland, Clippers, Cavaliers at Layer 11

In [ ]:
layer = 11

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=2, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
concept_an.gen_and_print([4416, 13040, 13098], [2, 2, 2], layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

## Pets Dogs and Cats

In [ ]:
text_idx = 4042
text = tr_analyzer.texts[text_idx]
prompt = tr_analyzer.corpus[text_idx]
text

In [ ]:
concept_an = ConceptAnalysis(prompt, tr_analyzer)

In [ ]:
concept_an.analyze_concepts()

In [ ]:
t_idcs = [2, 3]
t_idcs = [1]
# t_idcs = [2]
t_idcs = [8, 82]

### Pets Dogs and Cats at Layer 0

In [ ]:
layer = 0

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=0.2, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
# v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

### Pets Dogs and Cats at Layer 8

In [ ]:
layer = 8

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=0.2, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals[:10], idcs[:10]

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
if np.any(vals[1:] > 0):
    c = concept_an.gen_and_print(idcs[1:], vals[1:], layer, with_text=True, limit=10)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
# v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

### Pets Dogs and Cats at Layer 11

In [ ]:
layer = 11

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
# v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)